In [ ]:
df["geometry"] = gpd.GeoSeries.from_wkt(df["the_geom"])

ntas = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:2263")
ntas = ntas.to_crs("EPSG:4326")


In [ ]:
rent = pd.read_csv('medianAskingRent_Studio.csv')
rent

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt  

nta_df = pd.read_csv("2020_Neighborhood_Tabulation_Areas_(NTAs)_20251129.csv")
nta_df["geometry"] = nta_df["the_geom"].apply(wkt.loads)

ntas = gpd.GeoDataFrame(
    nta_df,
    geometry="geometry",
    crs="EPSG:4326",   # coords look like lon/lat, so WGS84 is fine
)

rent = pd.read_csv("medianAskingRent_OneBd.csv")

month_cols = [c for c in rent.columns if c[:4].isdigit()]
latest_col = sorted(month_cols)[-1]

rent_latest = rent[["areaName", "Borough", latest_col]].copy()
rent_latest = rent_latest.rename(columns={latest_col: "median_rent"})

rent_latest["neigh_clean"] = (
    rent_latest["areaName"]
    .str.lower()
    .str.strip()
)

ntas["neigh_clean"] = (
    ntas["NTAName"]
    .str.lower()
    .str.strip()
)

ntas_rent = ntas.merge(
    rent_latest[["neigh_clean", "median_rent"]],
    on="neigh_clean",
    how="left",
)

m = ntas_rent.explore(
    column="median_rent",
    cmap="viridis",
    scheme="Quantiles",=
    k=5,
    legend=True,
    tiles="CartoDB positron",
    tooltip=["NTAName", "BoroName", "median_rent"],=
)

m